# Build `nmdc_metadata.biosample_to_workflow_run`

Precomputes the (biosample, workflow run) mapping with provenance
flags for each MaterialProcessing type in the chain.
Covers **ALL** workflow types discovered in `workflow_execution_set`,
not just annotation runs.

See `docs/biosample_to_workflow_run.md` for schema, example queries, and
maintenance instructions — especially what to do when new MaterialProcessing
subclasses are added to the NMDC schema.

**Run this notebook after each NMDC data load.**

### Session setup

In [ ]:
import pandas as pd
import time

conn = get_trino_connection()
print("Trino connection ready")

### Configuration

**MAINTENANCE NOTE:** `PROCESSING_TYPES` is hardcoded against the MaterialProcessing
subclasses present in BERDL as of 2026-04-30. If a new subclass is added to the
NMDC schema and loaded into BERDL, add it here before rebuilding.

This notebook covers **all** workflow types discovered in `workflow_execution_set`.
A preflight cell below shows the full breakdown by type.

In [ ]:
PROCESSING_TYPES = {
    'nmdc:Extraction':                        'has_extraction',
    'nmdc:LibraryPreparation':                'has_library_prep',
    'nmdc:SubSamplingProcess':                'has_subsampling',
    'nmdc:Pooling':                           'has_pooling',
    'nmdc:ChromatographicSeparationProcess':  'has_chromatographic_separation',
    'nmdc:DissolvingProcess':                 'has_dissolving',
    'nmdc:ChemicalConversionProcess':         'has_chemical_conversion',
    'nmdc:FiltrationProcess':                 'has_filtration',
}

MAX_DEPTH     = 15
TARGET_SCHEMA = 'nmdc_metadata'
TARGET_TABLE  = 'biosample_to_workflow_run'

### Preflight: check for unknown MaterialProcessing types and show all workflow types

If any MaterialProcessing type appears here that is not in `PROCESSING_TYPES`, add it
to the config cell above before proceeding.

The workflow type breakdown is printed for discovery — all types will be walked.

In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT type, COUNT(*) AS n
    FROM   nmdc_metadata.material_processing_set
    GROUP  BY type
    ORDER  BY n DESC
""")
import pandas as pd
observed = pd.DataFrame(cur.fetchall(), columns=["type", "n"])

unknown = set(observed["type"]) - set(PROCESSING_TYPES.keys())
if unknown:
    print(f"WARNING: unknown MaterialProcessing types not in PROCESSING_TYPES:")
    for t in sorted(unknown):
        print(f"  {t}")
    print("Add them to PROCESSING_TYPES before continuing.")
else:
    print("OK: all MaterialProcessing types accounted for")
display(observed)

# Show all workflow types for discovery
spark_pf = get_spark_session(app_name='preflight', tenant_name='nmdc')
print("\nWorkflow types in workflow_execution_set:")
spark_pf.sql("""
    SELECT type, COUNT(*) AS n
    FROM   nmdc_metadata.workflow_execution_set
    GROUP  BY type
    ORDER  BY n DESC
""").toPandas()

### Graph edge table and iterative walk

`nmdc_metadata.graph_edges` is a precomputed 2-column table (src, next_id)
combining the four Silver provenance side tables. It is created once in this
notebook's Step 0 and must be refreshed whenever NMDC data is reloaded.

The walk is iterative — one flat JOIN per hop level — rather than recursive.
`WITH RECURSIVE` in Trino exceeds the 150-stage query planner limit for
datasets of this size regardless of batch configuration.

In [ ]:
def _walk_iterative(cur, workflow_ids, max_depth=MAX_DEPTH):
    """Walk from workflow run IDs upstream to biosamples via flat per-hop JOINs.

    Returns (bsm_df, mp_df):
      bsm_df: DataFrame(workflow_run_id, biosample_id, n_hops)
      mp_df:  DataFrame(workflow_run_id, processing_type) — deduplicated
    """
    frontier = pd.DataFrame({'origin': workflow_ids, 'id': workflow_ids})
    bsm_parts = []
    mp_parts  = []

    for depth in range(max_depth):
        if frontier.empty:
            break

        active_ids_sql = ', '.join(f"'{i}'" for i in frontier['id'].unique())

        # One hop: look up all outgoing edges from the current frontier
        cur.execute(f"""
            SELECT src, next_id
            FROM   nmdc_metadata.graph_edges
            WHERE  src IN ({active_ids_sql})
        """)
        edges = pd.DataFrame(cur.fetchall(), columns=['src', 'next_id'])
        if edges.empty:
            break

        next_step = (frontier
                     .merge(edges, left_on='id', right_on='src')
                     [['origin', 'next_id']]
                     .rename(columns={'next_id': 'id'}))

        # Collect biosample endpoints reached this hop
        bsm = next_step[next_step['id'].str.startswith('nmdc:bsm')].copy()
        if not bsm.empty:
            bsm['n_hops'] = depth + 1
            bsm_parts.append(bsm)

        # Continue frontier with non-biosample nodes
        frontier = next_step[~next_step['id'].str.startswith('nmdc:bsm')].drop_duplicates()

        # Resolve MaterialProcessing types for new frontier nodes
        if not frontier.empty:
            new_ids_sql = ', '.join(f"'{i}'" for i in frontier['id'].unique())
            cur.execute(f"""
                SELECT id, type
                FROM   nmdc_metadata.material_processing_set
                WHERE  id IN ({new_ids_sql})
            """)
            mp_rows = cur.fetchall()
            if mp_rows:
                mp_df = pd.DataFrame(mp_rows, columns=['id', 'type'])
                mp_step = (frontier
                           .merge(mp_df, on='id')
                           [['origin', 'type']]
                           .rename(columns={'origin': 'workflow_run_id', 'type': 'processing_type'})
                           .drop_duplicates())
                mp_parts.append(mp_step)

    bsm_out = (
        pd.concat(bsm_parts, ignore_index=True)
          .rename(columns={'origin': 'workflow_run_id', 'id': 'biosample_id'})
          .groupby(['workflow_run_id', 'biosample_id'], as_index=False)['n_hops'].min()
    ) if bsm_parts else pd.DataFrame(columns=['workflow_run_id', 'biosample_id', 'n_hops'])

    mp_out = (
        pd.concat(mp_parts, ignore_index=True).drop_duplicates()
    ) if mp_parts else pd.DataFrame(columns=['workflow_run_id', 'processing_type'])

    return bsm_out, mp_out

### Step 0: build/refresh `nmdc_metadata.graph_edges`

Small (87K rows) precomputed edge table combining the four Silver provenance
side tables. Refresh this whenever NMDC Silver tables are reloaded.

In [ ]:
spark = get_spark_session(app_name="build_biosample_to_workflow_run", tenant_name="nmdc")

spark.sql("""
    CREATE OR REPLACE TABLE nmdc_metadata.graph_edges
    USING DELTA AS
    SELECT parent_id AS src, was_informed_by AS next_id, 'was_informed_by' AS slot
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by
    UNION ALL
    SELECT parent_id AS src, has_input AS next_id, 'has_input' AS slot
    FROM   nmdc_metadata.data_generation_set_has_input
    UNION ALL
    SELECT has_output AS src, parent_id AS next_id, 'has_output' AS slot
    FROM   nmdc_metadata.material_processing_set_has_output
    UNION ALL
    SELECT parent_id AS src, has_input AS next_id, 'has_input' AS slot
    FROM   nmdc_metadata.material_processing_set_has_input
""")
n = spark.sql("SELECT COUNT(*) FROM nmdc_metadata.graph_edges").collect()[0][0]
print(f"graph_edges: {n:,} rows")

### Step 1: collect ALL workflow runs

In [ ]:
t0 = time.monotonic()
cur = conn.cursor()
cur.execute("""
    SELECT id, type
    FROM   nmdc_metadata.workflow_execution_set
""")
wf_df = pd.DataFrame(cur.fetchall(), columns=["workflow_run_id", "workflow_type"])
print(f"{len(wf_df)} workflow runs  ({time.monotonic()-t0:.1f}s)")
wf_df["workflow_type"].value_counts()

### Step 2: iterative walk — all workflow runs in one pass

In [ ]:
r2b, mp_all = _walk_iterative(cur, wf_df["workflow_run_id"].tolist())
print(f"{len(r2b)} (workflow_run_id, biosample_id) pairs")
print(f"{len(mp_all)} (workflow_run_id, processing_type) hits")
print(f"({time.monotonic()-t0:.1f}s elapsed)")

### Step 3: pivot processing types to boolean columns

In [ ]:
mp_pivot = (
    mp_all.assign(flag=True)
          .pivot_table(index="workflow_run_id", columns="processing_type",
                       values="flag", aggfunc="any", fill_value=False)
          .reset_index()
)
mp_pivot.columns.name = None
mp_pivot = mp_pivot.rename(columns=PROCESSING_TYPES)
for col in PROCESSING_TYPES.values():
    if col not in mp_pivot.columns:
        mp_pivot[col] = False

result = (
    r2b
    .merge(wf_df, on="workflow_run_id", how="left")
    .merge(mp_pivot[["workflow_run_id"] + list(PROCESSING_TYPES.values())],
           on="workflow_run_id", how="left")
)
for col in PROCESSING_TYPES.values():
    result[col] = result[col].fillna(False)

bool_cols = list(PROCESSING_TYPES.values())
result = result[["biosample_id", "workflow_run_id", "workflow_type", "n_hops"] + bool_cols]

print(f"{len(result)} rows, {len(result.columns)} columns")
result.head()

### Step 4: write directly to Silver as a Delta table

In [ ]:
(spark.createDataFrame(result)
      .write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{TARGET_SCHEMA}.{TARGET_TABLE}"))

print(f"Wrote {TARGET_SCHEMA}.{TARGET_TABLE}")
print(f"Total build time: {time.monotonic()-t0:.1f}s")

### Step 5: verify

In [ ]:
spark.sql(f"""
    SELECT workflow_type,
           COUNT(DISTINCT biosample_id) AS n_biosamples,
           COUNT(DISTINCT workflow_run_id) AS n_workflow_runs,
           ROUND(AVG(n_hops), 1) AS avg_hops,
           SUM(CAST(has_pooling AS INT)) AS pooled_pairs,
           SUM(CAST(has_extraction AS INT)) AS extracted_pairs
    FROM   {TARGET_SCHEMA}.{TARGET_TABLE}
    GROUP  BY workflow_type
    ORDER  BY n_workflow_runs DESC
""").toPandas()

### Cleanup: drop old table

If `biosample_to_annotation_run` still exists from a prior run, drop it now.

In [ ]:
spark.sql('DROP TABLE IF EXISTS nmdc_metadata.biosample_to_annotation_run')
print('Dropped nmdc_metadata.biosample_to_annotation_run (if it existed)')